<a href="https://colab.research.google.com/github/nuhuynhh/AAI2026/blob/main/inventory_replenishment_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inventory Replenishment Agent

---

---
## 1. Setup — Imports and Global Parameters

In [ ]:
import pandas as pd
import numpy as np
from collections import deque

# ── Global configuration ─────────────────────────────────────────────────────
ALPHA           = 0.3   # EWMA smoothing factor (0 < α ≤ 1; higher = more reactive)
ORDER_COST_FIXED = 5.0  # Fixed cost per purchase order placed (set 0 to disable)
MAX_COVER_DAYS  = 30    # Inventory cap: agent will not order beyond this many days
                        # of expected demand → prevents unrealistic over-ordering
SEED            = 42
np.random.seed(SEED)

# ── File paths ────────────────────────────────────────────────────────────────
SALES_PATH  = "sales.csv"
INV_PATH    = "inventory.csv"
PARAMS_PATH = "params.csv"

---
## 2. Data Generation (Demo)

> **Skip this cell** if you already have `sales.csv`, `inventory.csv`, and `params.csv` in the working directory.

In [ ]:
def generate_demo_data(n_days: int = 90) -> None:
    """Write three demo CSV files so the notebook is self-contained."""
    dates = pd.date_range("2025-01-01", periods=n_days, freq="D")
    skus  = ["A100", "B200", "C300"]

    sales_rows = []
    for sku in skus:
        base   = {"A100": 20, "B200": 12, "C300": 8}[sku]
        amp    = {"A100": 4,  "B200": 2,  "C300": 1}[sku]
        season = np.sin(np.linspace(0, 3 * np.pi, n_days)) * amp
        noise  = np.random.normal(0, 3, n_days)
        demand = np.maximum(0, np.round(base + season + noise)).astype(int)
        for d, q in zip(dates, demand):
            sales_rows.append({"date": d.strftime("%Y-%m-%d"), "sku": sku, "qty_sold": int(q)})

    pd.DataFrame(sales_rows).to_csv(SALES_PATH, index=False)

    pd.DataFrame([
        {"sku": "A100", "opening_stock": 250},
        {"sku": "B200", "opening_stock": 180},
        {"sku": "C300", "opening_stock": 100},
    ]).to_csv(INV_PATH, index=False)

    pd.DataFrame([
        {"sku": "A100", "unit_cost": 10, "holding_cost_per_day": 0.02,
         "stockout_cost": 2.0, "lead_time_days": 5,  "min_order_qty": 50, "service_level": 0.95},
        {"sku": "B200", "unit_cost": 8,  "holding_cost_per_day": 0.015,
         "stockout_cost": 1.5, "lead_time_days": 7,  "min_order_qty": 40, "service_level": 0.95},
        {"sku": "C300", "unit_cost": 5,  "holding_cost_per_day": 0.01,
         "stockout_cost": 1.0, "lead_time_days": 3,  "min_order_qty": 30, "service_level": 0.90},
    ]).to_csv(PARAMS_PATH, index=False)

    print(f"Demo data written: {n_days} days, SKUs = A100 / B200 / C300")


generate_demo_data()

Demo data written: 90 days, SKUs = A100 / B200 / C300


---
## 3. Data Loading and Validation

In [ ]:
def load_and_validate(
    sales_path: str = SALES_PATH,
    inv_path:   str = INV_PATH,
    params_path: str = PARAMS_PATH,
) -> tuple:
    """
    Load the three input CSVs and validate their schemas and key integrity.
    Returns (sales_df, inv_df, params_df).
    Raises ValueError with a descriptive message on any problem.
    """
    # ── Load ─────────────────────────────────────────────────────────────────
    sales  = pd.read_csv(sales_path,  parse_dates=["date"])
    inv    = pd.read_csv(inv_path)
    params = pd.read_csv(params_path)

    # ── Schema checks ─────────────────────────────────────────────────────────
    required = {
        "sales":  {"date", "sku", "qty_sold"},
        "inv":    {"sku", "opening_stock"},
        "params": {"sku", "unit_cost", "holding_cost_per_day", "stockout_cost",
                   "lead_time_days", "min_order_qty", "service_level"},
    }
    for name, cols in required.items():
        df = {"sales": sales, "inv": inv, "params": params}[name]
        missing = cols - set(df.columns)
        if missing:
            raise ValueError(f"{name}.csv is missing columns: {missing}")

    # ── Key integrity: every SKU in sales must appear in params and inventory ─
    sales_skus  = set(sales["sku"].unique())
    params_skus = set(params["sku"].unique())
    inv_skus    = set(inv["sku"].unique())

    no_params = sales_skus - params_skus
    no_inv    = sales_skus - inv_skus
    if no_params:
        raise ValueError(f"SKUs in sales with no params entry: {no_params}")
    if no_inv:
        raise ValueError(f"SKUs in sales with no inventory entry: {no_inv}")

    # ── Basic sanity checks ───────────────────────────────────────────────────
    if (sales["qty_sold"] < 0).any():
        raise ValueError("Negative qty_sold values found in sales.csv")
    if (params["service_level"] > 1).any() or (params["service_level"] <= 0).any():
        raise ValueError("service_level must be in (0, 1]")

    print(f"✓ sales.csv     : {len(sales):,} rows | date range {sales['date'].min().date()} → {sales['date'].max().date()}")
    print(f"✓ inventory.csv : {len(inv)} SKUs")
    print(f"✓ params.csv    : {len(params)} SKUs")
    print(f"  SKUs found    : {sorted(sales_skus)}")
    return sales, inv, params


sales_raw, inv_df, params_df = load_and_validate()

✓ sales.csv     : 270 rows | date range 2025-01-01 → 2025-03-31
✓ inventory.csv : 3 SKUs
✓ params.csv    : 3 SKUs
  SKUs found    : ['A100', 'B200', 'C300']


---
## 4. Demand Aggregation — Daily Demand per SKU

In [ ]:
def aggregate_daily_demand(sales: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate transaction-level sales to one row per (date, sku).
    Fill gaps (dates where an SKU had zero sales) with 0 so every SKU
    has a complete, contiguous daily calendar — important for EWMA continuity.
    """
    # Sum within the same (date, sku) — handles multi-transaction days
    daily = (
        sales
        .groupby(["date", "sku"], as_index=False)["qty_sold"]
        .sum()
    )

    # Build a complete date × SKU grid and fill missing entries with 0
    full_dates = pd.date_range(daily["date"].min(), daily["date"].max(), freq="D")
    all_skus   = daily["sku"].unique()
    grid = pd.MultiIndex.from_product([full_dates, all_skus], names=["date", "sku"])

    daily = (
        daily
        .set_index(["date", "sku"])
        .reindex(grid, fill_value=0)
        .reset_index()
    )
    daily = daily.sort_values(["sku", "date"]).reset_index(drop=True)
    return daily


sales = aggregate_daily_demand(sales_raw)

print(f"Daily demand table: {len(sales):,} rows ({sales['sku'].nunique()} SKUs × {sales['date'].nunique()} days)")
sales.head(8)

Daily demand table: 270 rows (3 SKUs × 90 days)


,date,sku,qty_sold
0,2025-01-01,A100,21
1,2025-01-02,A100,20
2,2025-01-03,A100,23
3,2025-01-04,A100,26
4,2025-01-05,A100,21
5,2025-01-06,A100,21
6,2025-01-07,A100,27
7,2025-01-08,A100,25


---
## 5. EWMA Forecasting

The Exponentially Weighted Moving Average assigns exponentially decaying weights to past observations — more recent days matter more. The update rule is:

$$\hat{d}_t = \alpha \cdot d_{t-1} + (1-\alpha) \cdot \hat{d}_{t-1}$$

where $\alpha \in (0,1]$ controls how quickly the forecast adapts.  
The forecast for day $t$ is produced *before* day $t$'s demand is revealed, so it is a true one-step-ahead prediction.

In [ ]:
def ewma_forecast_series(demand: pd.Series, alpha: float = ALPHA) -> tuple:
    """
    Compute one-step-ahead EWMA forecasts for a single SKU's daily demand series.

    The forecast on day t uses information up to day t-1 only:
        forecast[t] = alpha * demand[t-1] + (1-alpha) * forecast[t-1]

    Parameters
    ----------
    demand : pd.Series  — observed daily quantities (chronological order)
    alpha  : float      — EWMA smoothing factor

    Returns
    -------
    forecasts : pd.Series  — predicted demand for each day (same index as demand)
    errors    : pd.Series  — forecast error = demand − forecast (for safety stock σ)
    """
    forecasts = np.empty(len(demand))

    # Seed: use the first observed value as the initial forecast
    forecasts[0] = demand.iloc[0]

    # Update: forecast[t] is based on what we knew through t-1
    for t in range(1, len(demand)):
        forecasts[t] = alpha * demand.iloc[t - 1] + (1 - alpha) * forecasts[t - 1]

    forecasts = pd.Series(forecasts, index=demand.index)
    errors    = demand - forecasts  # positive error = under-forecast
    return forecasts, errors


# Quick sanity check on one SKU
sku_check = sales[sales["sku"] == sales["sku"].iloc[0]].copy()
fc_check, err_check = ewma_forecast_series(sku_check["qty_sold"], alpha=ALPHA)
print(f"EWMA sanity check — mean forecast: {fc_check.mean():.2f}, mean |error|: {err_check.abs().mean():.2f}")

EWMA sanity check — mean forecast: 20.50, mean |error|: 2.49


---
## 6. Safety Stock and Reorder Point

Safety stock provides a buffer against demand variability during the lead-time period:

$$SS = z(\text{service\_level}) \times \sigma_{\text{error}} \times \sqrt{\text{lead\_time}}$$

The reorder point (ROP) is the inventory level at which we must order to avoid a stockout:

$$ROP = \hat{d} \times \text{lead\_time} + SS$$

$\sigma_{\text{error}}$ is estimated from the expanding standard deviation of forecast errors, so it improves as more data is seen.

In [ ]:
def z_from_service_level(p: float) -> float:
    """
    Map a service-level probability to the standard-normal z-score.
    Uses a lookup table for common values with linear interpolation.
    """
    table = {
        0.80: 0.84, 0.85: 1.04, 0.90: 1.28,
        0.95: 1.65, 0.97: 1.88, 0.98: 2.05, 0.99: 2.33,
    }
    keys = sorted(table)
    if p in table:
        return table[p]
    if p <= keys[0]:
        return table[keys[0]]
    if p >= keys[-1]:
        return table[keys[-1]]
    # Linear interpolation between bracketing keys
    for lo, hi in zip(keys, keys[1:]):
        if lo < p < hi:
            w = (p - lo) / (hi - lo)
            return table[lo] * (1 - w) + table[hi] * w


def compute_safety_stock(errors: pd.Series, lead_time: int, z: float) -> pd.Series:
    """
    Compute expanding safety stock for each day.

    Uses the expanding (cumulative) standard deviation of forecast errors
    so the safety stock estimate improves over time as more error history
    accumulates.  A minimum floor of 0 is enforced.

    Parameters
    ----------
    errors    : pd.Series — forecast errors (demand − forecast)
    lead_time : int       — replenishment lead time in days
    z         : float     — service-level z-score

    Returns
    -------
    safety_stock : pd.Series — day-by-day safety stock values
    """
    # Expanding std: on day 0 there is no history, so fill with global std
    expanding_std = errors.expanding(min_periods=2).std()
    global_std    = errors.std() if not np.isnan(errors.std()) else 0.0
    expanding_std = expanding_std.fillna(global_std).clip(lower=0.0)

    safety_stock = z * expanding_std * np.sqrt(max(1, lead_time))
    return safety_stock.fillna(0.0)

---
## 7. Replenishment Agent — Core Decision Logic

In [ ]:
def replenishment_decision(
    on_hand:        float,
    pipeline_qty:   float,
    forecast_today: float,
    safety_stock:   float,
    lead_time:      int,
    min_order_qty:  int,
    avg_demand:     float,
    max_cover_days: int = MAX_COVER_DAYS,
) -> tuple:
    """
    Decide whether to place a purchase order today and, if so, how much.

    Logic
    -----
    1. Compute the reorder point (ROP) from lead-time demand + safety stock.
    2. Compute the inventory position (on-hand + already-ordered pipeline).
    3. If position < ROP → order up to the target level.
    4. Enforce min_order_qty and inventory cap (guardrail).

    Returns
    -------
    order_qty : int   — units to order (0 = no order)
    rop       : float — reorder point used in the decision
    target    : float — order-up-to target level
    reason    : str   — plain-English rationale for the log
    """
    daily_rate   = max(0.0, forecast_today)
    rop          = lead_time * daily_rate + safety_stock

    # Order-up-to target = cover lead time + safety buffer + 1 review day
    target = rop + max(1.0, daily_rate)

    # Inventory cap: never plan to hold more than MAX_COVER_DAYS of demand
    cap    = max_cover_days * max(1.0, avg_demand)
    target = min(target, cap)

    # Inventory position accounts for open orders in transit (pipeline)
    position = on_hand + pipeline_qty

    if position >= rop:
        return 0, rop, target, f"position={position:.1f} ≥ ROP={rop:.1f} → WAIT"

    raw_qty   = target - position
    order_qty = max(min_order_qty, int(np.ceil(raw_qty)))

    reason = (
        f"position={position:.1f} < ROP={rop:.1f} "
        f"[forecast={daily_rate:.2f}, SS={safety_stock:.2f}] → "
        f"ORDER {order_qty} (MOQ={min_order_qty}, cap={cap:.0f})"
    )
    return order_qty, rop, target, reason

---
## 8. Day-by-Day Simulation

In [ ]:
def simulate_agent(
    sales:   pd.DataFrame,
    inv:     pd.DataFrame,
    params:  pd.DataFrame,
    alpha:   float = ALPHA,
    order_cost_fixed: float = ORDER_COST_FIXED,
    max_cover_days: int = MAX_COVER_DAYS,
    verbose: bool = True,
) -> tuple:
    """
    Run the replenishment agent for every SKU over the full simulation horizon.

    For each SKU, the loop proceeds day by day:
        1. Receive any POs due today.
        2. Serve demand; record lost sales (stockouts).
        3. Charge holding and stockout costs.
        4. Update EWMA forecast and safety stock.
        5. Decide whether to place a new PO.
        6. Log the full decision with rationale.

    Returns
    -------
    summary : pd.DataFrame — one row per SKU with aggregate metrics
    logs    : list[str]    — daily decision log entries
    details : list[dict]   — day-level records for further analysis
    """
    pmap   = {r["sku"]: r.to_dict() for _, r in params.iterrows()}
    imap   = {r["sku"]: float(r["opening_stock"]) for _, r in inv.iterrows()}

    summary_rows = []
    all_logs     = []
    all_details  = []

    for sku in sorted(sales["sku"].unique()):
        # ── Per-SKU setup ─────────────────────────────────────────────────────
        s         = sales[sales["sku"] == sku].sort_values("date").reset_index(drop=True)
        qty_series = s["qty_sold"].astype(float)

        p           = pmap[sku]
        lead        = int(p["lead_time_days"])
        moq         = int(p["min_order_qty"])
        hold_rate   = float(p["holding_cost_per_day"])
        stk_rate    = float(p["stockout_cost"])
        service     = float(p["service_level"])
        z           = z_from_service_level(service)

        # Pre-compute full EWMA series and safety stock series
        forecasts, errors = ewma_forecast_series(qty_series, alpha=alpha)
        safety_stocks     = compute_safety_stock(errors, lead, z)

        # ── State variables ───────────────────────────────────────────────────
        on_hand      = imap.get(sku, 0.0)
        pipeline     = deque()   # entries: (arrival_day_index, qty)

        # Accumulators
        total_holding  = 0.0
        total_stockout = 0.0
        total_po_cost  = 0.0
        total_demanded = 0.0
        total_shipped  = 0.0
        stockout_days  = 0

        # Rolling average for inventory cap
        avg_demand_so_far = qty_series.mean()  # global average used as initialiser

        all_logs.append(f"\n{'='*70}")
        all_logs.append(f"SKU: {sku} | lead={lead}d | MOQ={moq} | service={service:.0%} (z={z:.2f})")
        all_logs.append(f"{'='*70}")

        for i, row in s.iterrows():
            date    = row["date"]
            demand  = float(row["qty_sold"])
            fc_today = float(forecasts.iloc[i])
            ss_today = float(safety_stocks.iloc[i])

            # ── 1. Receive POs arriving today ─────────────────────────────────
            received = 0.0
            while pipeline and pipeline[0][0] == i:
                arrived   = pipeline.popleft()[1]
                on_hand  += arrived
                received += arrived

            # ── 2. Serve demand ───────────────────────────────────────────────
            shipped    = min(on_hand, demand)
            lost_sales = max(0.0, demand - shipped)
            on_hand   -= shipped

            if lost_sales > 0:
                stockout_days += 1

            # ── 3. Compute daily costs ────────────────────────────────────────
            holding_cost  = on_hand * hold_rate
            stockout_cost = lost_sales * stk_rate
            daily_cost    = holding_cost + stockout_cost

            total_holding  += holding_cost
            total_stockout += stockout_cost
            total_demanded += demand
            total_shipped  += shipped

            # ── 4. Replenishment decision ─────────────────────────────────────
            # Pipeline qty = units already on order but not yet arrived
            pipeline_qty = sum(q for _, q in pipeline)

            # Update rolling avg demand for inventory cap
            if i > 0:
                avg_demand_so_far = qty_series.iloc[: i + 1].mean()

            order_qty, rop, target, reason = replenishment_decision(
                on_hand        = on_hand,
                pipeline_qty   = pipeline_qty,
                forecast_today = fc_today,
                safety_stock   = ss_today,
                lead_time      = lead,
                min_order_qty  = moq,
                avg_demand     = avg_demand_so_far,
                max_cover_days = max_cover_days,
            )

            if order_qty > 0:
                arrival_idx = i + lead
                pipeline.append((arrival_idx, order_qty))
                total_po_cost += order_cost_fixed
                daily_cost    += order_cost_fixed

            # ── 5. Log ────────────────────────────────────────────────────────
            log_line = (
                f"{date.date()} [{sku}] "
                f"demand={demand:.0f} shipped={shipped:.0f} lost={lost_sales:.0f} "
                f"on_hand={on_hand:.1f} | {reason}"
            )
            if received > 0:
                log_line = f"  ↳ received {received:.0f} units  " + log_line
            all_logs.append(log_line)

            all_details.append({
                "date":         date,
                "sku":          sku,
                "demand":       demand,
                "forecast":     fc_today,
                "safety_stock": ss_today,
                "shipped":      shipped,
                "lost_sales":   lost_sales,
                "on_hand":      on_hand,
                "order_qty":    order_qty,
                "holding_cost": holding_cost,
                "stockout_cost": stockout_cost,
                "daily_cost":   daily_cost,
            })

        # ── Per-SKU summary ───────────────────────────────────────────────────
        n_days     = len(s)
        fill_rate  = total_shipped / max(1.0, total_demanded)
        total_cost = total_holding + total_stockout + total_po_cost

        summary_rows.append({
            "sku":              sku,
            "period_days":      n_days,
            "total_demand":     int(total_demanded),
            "units_shipped":    int(total_shipped),
            "fill_rate":        round(fill_rate, 4),
            "stockout_days":    stockout_days,
            "ending_on_hand":   round(on_hand, 1),
            "holding_cost":     round(total_holding, 2),
            "stockout_cost":    round(total_stockout, 2),
            "po_fixed_cost":    round(total_po_cost, 2),
            "total_cost":       round(total_cost, 2),
            "avg_daily_cost":   round(total_cost / n_days, 2),
        })

    summary = pd.DataFrame(summary_rows)
    return summary, all_logs, all_details

### 9. Run the Agent

In [ ]:
summary_agent, logs_agent, details_agent = simulate_agent(
    sales, inv_df, params_df, alpha=ALPHA, verbose=True
)

print("\n" + "="*70)
print("AGENT SUMMARY")
print("="*70)
print(summary_agent.to_string(index=False))


AGENT SUMMARY
 sku  period_days  total_demand  units_shipped  fill_rate  stockout_days  ending_on_hand  holding_cost  stockout_cost  po_fixed_cost  total_cost  avg_daily_cost
A100           90          1847           1847     1.0000              0            53.0         84.10            0.0          175.0      259.10            2.88
B200           90          1138           1130     0.9930              3            10.0         56.70           12.0          135.0      203.70            2.26
C300           90           740            734     0.9919              1            26.0         23.61            6.0          115.0      144.61            1.61


### 9.1 Print the Daily Agent Log

In [ ]:
# Print first 60 lines of the log; remove the cap to see everything
LOG_LINES_TO_SHOW = 60
for line in logs_agent[:LOG_LINES_TO_SHOW]:
    print(line)


SKU: A100 | lead=5d | MOQ=50 | service=95% (z=1.65)
2025-01-01 [A100] demand=21 shipped=21 lost=0 on_hand=229.0 | position=229.0 ≥ ROP=116.7 → WAIT
2025-01-02 [A100] demand=20 shipped=20 lost=0 on_hand=209.0 | position=209.0 ≥ ROP=107.6 → WAIT
2025-01-03 [A100] demand=23 shipped=23 lost=0 on_hand=186.0 | position=186.0 ≥ ROP=109.7 → WAIT
2025-01-04 [A100] demand=26 shipped=26 lost=0 on_hand=160.0 | position=160.0 ≥ ROP=116.2 → WAIT
2025-01-05 [A100] demand=21 shipped=21 lost=0 on_hand=139.0 | position=139.0 ≥ ROP=123.5 → WAIT
2025-01-06 [A100] demand=21 shipped=21 lost=0 on_hand=118.0 | position=118.0 < ROP=120.4 [forecast=22.24, SS=9.16] → ORDER 50 (MOQ=50, cap=660)
2025-01-07 [A100] demand=27 shipped=27 lost=0 on_hand=91.0 | position=141.0 ≥ ROP=119.9 → WAIT
2025-01-08 [A100] demand=25 shipped=25 lost=0 on_hand=66.0 | position=116.0 < ROP=126.9 [forecast=23.41, SS=9.81] → ORDER 50 (MOQ=50, cap=690)
2025-01-09 [A100] demand=22 shipped=22 lost=0 on_hand=44.0 | position=144.0 ≥ ROP=129

---
## 10. Evaluation Metrics

In [ ]:
def compute_metrics(summary: pd.DataFrame) -> pd.DataFrame:
    """
    Display the key evaluation metrics clearly.

    Metrics
    -------
    fill_rate      : fraction of demand fulfilled (higher = better service)
    stockout_days  : number of days with any unmet demand (lower = better)
    holding_cost   : total cost of carrying unsold inventory
    stockout_cost  : total cost of unmet demand (lost sales penalty)
    total_cost     : holding + stockout + fixed PO costs
    """
    metrics = summary[[
        "sku", "total_demand", "units_shipped", "fill_rate",
        "stockout_days", "holding_cost", "stockout_cost",
        "po_fixed_cost", "total_cost", "avg_daily_cost"
    ]].copy()
    metrics["fill_rate_pct"] = (metrics["fill_rate"] * 100).round(2).astype(str) + " %"
    return metrics


metrics_agent = compute_metrics(summary_agent)
print("\n── AGENT EVALUATION METRICS ──")
print(metrics_agent.to_string(index=False))


── AGENT EVALUATION METRICS ──
 sku  total_demand  units_shipped  fill_rate  stockout_days  holding_cost  stockout_cost  po_fixed_cost  total_cost  avg_daily_cost fill_rate_pct
A100          1847           1847     1.0000              0         84.10            0.0          175.0      259.10            2.88       100.0 %
B200          1138           1130     0.9930              3         56.70           12.0          135.0      203.70            2.26        99.3 %
C300           740            734     0.9919              1         23.61            6.0          115.0      144.61            1.61       99.19 %


---
## 11. Baseline Strategy

The **fixed-cycle baseline** places an order of exactly `min_order_qty` units on a fixed schedule every `lead_time + 1` days, with no demand forecast and no safety stock. This is a common "naive" rule used in practice when forecasting capability is absent.

Comparing the agent against this baseline isolates the value added by EWMA forecasting and service-level safety stock.

In [ ]:
def simulate_baseline(
    sales:   pd.DataFrame,
    inv:     pd.DataFrame,
    params:  pd.DataFrame,
    order_cost_fixed: float = ORDER_COST_FIXED,
) -> pd.DataFrame:
    """
    Fixed-cycle baseline: order exactly min_order_qty every (lead_time + 1) days.
    No forecast, no safety stock, no inventory position logic.
    """
    pmap = {r["sku"]: r.to_dict() for _, r in params.iterrows()}
    imap = {r["sku"]: float(r["opening_stock"]) for _, r in inv.iterrows()}

    rows = []
    for sku in sorted(sales["sku"].unique()):
        s      = sales[sales["sku"] == sku].sort_values("date").reset_index(drop=True)
        p      = pmap[sku]
        lead   = int(p["lead_time_days"])
        moq    = int(p["min_order_qty"])
        cycle  = lead + 1            # order every (lead+1) days

        on_hand       = imap.get(sku, 0.0)
        pipeline      = deque()       # (arrival_idx, qty)
        total_holding  = 0.0
        total_stockout = 0.0
        total_po_cost  = 0.0
        total_demanded = 0.0
        total_shipped  = 0.0
        stockout_days  = 0

        for i, row in s.iterrows():
            demand = float(row["qty_sold"])

            # Receive due POs
            while pipeline and pipeline[0][0] == i:
                on_hand += pipeline.popleft()[1]

            # Serve demand
            shipped    = min(on_hand, demand)
            lost_sales = max(0.0, demand - shipped)
            on_hand   -= shipped

            if lost_sales > 0:
                stockout_days += 1

            total_holding  += on_hand * float(p["holding_cost_per_day"])
            total_stockout += lost_sales * float(p["stockout_cost"])
            total_demanded += demand
            total_shipped  += shipped

            # Fixed-cycle order trigger
            if i % cycle == 0:
                pipeline.append((i + lead, moq))
                total_po_cost += order_cost_fixed

        total_cost = total_holding + total_stockout + total_po_cost
        fill_rate  = total_shipped / max(1.0, total_demanded)
        rows.append({
            "sku":           sku,
            "fill_rate":     round(fill_rate, 4),
            "stockout_days": stockout_days,
            "holding_cost":  round(total_holding, 2),
            "stockout_cost": round(total_stockout, 2),
            "total_cost":    round(total_cost, 2),
        })
    return pd.DataFrame(rows)


summary_baseline = simulate_baseline(sales, inv_df, params_df)

### 11.1 Comparison Table

In [ ]:
def build_comparison_table(
    agent_summary:    pd.DataFrame,
    baseline_summary: pd.DataFrame,
) -> pd.DataFrame:
    """
    Side-by-side comparison of the agent vs the fixed-cycle baseline.
    Also computes the absolute and relative improvement for each KPI.
    """
    a = agent_summary[[
        "sku", "fill_rate", "stockout_days", "holding_cost", "stockout_cost", "total_cost"
    ]].rename(columns=lambda c: f"agent_{c}" if c != "sku" else c)

    b = baseline_summary.rename(columns=lambda c: f"base_{c}" if c != "sku" else c)

    cmp = a.merge(b, on="sku")

    # Improvement columns (positive = agent is better)
    cmp["Δ_fill_rate"]     = (cmp["agent_fill_rate"]     - cmp["base_fill_rate"]).round(4)
    cmp["Δ_stockout_days"] = (cmp["base_stockout_days"]  - cmp["agent_stockout_days"])   # fewer is better
    cmp["Δ_total_cost"]    = (cmp["base_total_cost"]      - cmp["agent_total_cost"]).round(2)

    return cmp


comparison = build_comparison_table(summary_agent, summary_baseline)

print("\n── AGENT vs BASELINE COMPARISON ──")
print(comparison.to_string(index=False))

print("\n── IMPROVEMENT SUMMARY (positive = agent wins) ──")
for _, r in comparison.iterrows():
    print(
        f"  {r['sku']}: fill rate {r['Δ_fill_rate']:+.2%}  "
        f"| fewer stockout days {r['Δ_stockout_days']:+.0f}  "
        f"| cost savings ${r['Δ_total_cost']:+.2f}"
    )


── AGENT vs BASELINE COMPARISON ──
 sku  agent_fill_rate  agent_stockout_days  agent_holding_cost  agent_stockout_cost  agent_total_cost  base_fill_rate  base_stockout_days  base_holding_cost  base_stockout_cost  base_total_cost  Δ_fill_rate  Δ_stockout_days  Δ_total_cost
A100           1.0000                    0               84.10                  0.0            259.10          0.5263                  48              44.50              1750.0          1869.50       0.4737               48       1610.40
B200           0.9930                    3               56.70                 12.0            203.70          0.5448                  45              31.74               777.0           868.74       0.4482               42        665.04
C300           0.9919                    1               23.61                  6.0            144.61          1.0000                   0              45.27                 0.0           160.27      -0.0081               -1         15.66

── IMPROVEM